### Exercise 1



Assume that the size of the convolution kernel is $\Delta=0$. Show that in this case the convolution kernel implements an MLP independently for each set of channels. This leads to the Network in Network architectures (Lin et al., 2013).

Let's start from the general multi-channel convolutional layer definition from the notes:

$$[\mathsf{H}]_{i,j,d} = \sum_{a = -\Delta}^{\Delta} \sum_{b = -\Delta}^{\Delta} \sum_c [\mathsf{V}]_{a, b, c, d} [\mathsf{X}]_{i+a, j+b, c}$$

#### Setting Δ=0

When $\Delta = 0$, the sums over $a$ and $b$ collapse to a single term — only $a=0, b=0$ survives:

$$[\mathsf{H}]_{i,j,d} = \sum_c [\mathsf{V}]_{0, 0, c, d} \cdot [\mathsf{X}]_{i, j, c}$$

Let's clean up notation by writing $W_{c,d} := [\mathsf{V}]_{0,0,c,d}$, giving:

$$[\mathsf{H}]_{i,j,d} = \sum_c W_{c,d} \cdot [\mathsf{X}]_{i,j,c}$$

#### What does this say?

At each fixed spatial location $(i,j)$, this is exactly a **linear map** from the input channel vector to the output channel vector:

$$\mathbf{h}_{i,j} = W \, \mathbf{x}_{i,j}$$

where $\mathbf{x}_{i,j} \in \mathbb{R}^{C_{in}}$ is the vector of input channels at location $(i,j)$, and $W \in \mathbb{R}^{C_{out} \times C_{in}}$ is a weight matrix **shared across all spatial locations**.

This is precisely a **fully connected (linear) layer applied independently and identically to each spatial location's channel vector** — with no mixing of information across positions.

#### Why this is an MLP

Stacking multiple such $\Delta=0$ layers with nonlinearities between them gives, at each position $(i,j)$:

$$\mathbf{h}^{(1)}_{i,j} = \sigma\!\left(W^{(1)}\mathbf{x}_{i,j}\right), \quad \mathbf{h}^{(2)}_{i,j} = \sigma\!\left(W^{(2)}\mathbf{h}^{(1)}_{i,j}\right), \quad \ldots$$

This is a **full MLP acting on the channel dimension**, applied **pixel-wise** (independently at every spatial location). Spatially, the weights are still shared — so it's parameter-efficient — but in the channel dimension, you have the full expressive power of an MLP.

#### Connection to Network in Network (NiN)

This is exactly the **Network in Network** idea (Lin et al., 2013). Rather than using a plain linear projection across channels, NiN replaces the $1 \times 1$ convolution (which is what $\Delta=0$ gives you geometrically — a kernel of size $1\times 1$) with a small MLP across channels. The key insight is:

| Property | Standard conv ($\Delta > 0$) | $\Delta = 0$ (1×1 conv / NiN) |
|---|---|---|
| Spatial mixing | Yes, within $\Delta$-neighborhood | No — purely pointwise |
| Channel mixing | Yes | Yes — full MLP |
| Parameters | $O(\Delta^2 \cdot C_{in} \cdot C_{out})$ | $O(C_{in} \cdot C_{out})$ |

So $\Delta = 0$ kernels are a powerful tool for **channel-wise feature transformation** without any spatial cost, and stacking them gives you a learned, nonlinear mixing of channels at every pixel independently.

### Exercise 2

Audio data is often represented as a one-dimensional sequence.
1. When might you want to impose locality and translation invariance for audio?
2. Derive the convolution operations for audio.
3. Can you treat audio using the same tools as computer vision? Hint: use the spectrogram.

#### Part 1: When do locality and translation invariance make sense for audio?

**Translation invariance** makes sense whenever the identity of a pattern doesn't depend on *when* it occurs in time. For example:

- A spoken word ("cat") should be recognized whether it appears at the beginning or middle of a recording
- An environmental sound (a doorbell, a cough) is the same sound wherever it sits in the timeline

**Locality** makes sense because meaningful acoustic patterns — a phoneme, a transient, a beat — are composed of nearby samples in time. To decide what's happening at time $t$, you only need a short window of surrounding samples, not the entire recording. Distant parts of the signal are irrelevant to the local pattern.

That said, there are limits. Long-range temporal dependencies *do* matter for higher-level understanding (grammar in speech, musical phrases, narrative), just as deeper CNN layers capture larger image structure. So locality is most appropriate at **lower layers**, with longer-range structure handled by deeper or recurrent layers.

---

#### Part 2: Deriving the 1D Convolution for Audio

Audio is a one-dimensional sequence, so we index the input as $[\mathbf{X}]_i$ where $i$ is the time step. The hidden representation is likewise $[\mathbf{H}]_i$.

Starting from a fully connected layer parametrized by a weight matrix, by analogy with the 2D derivation in the notes:

**Step 1 — General form:**
$$[\mathbf{H}]_i = u_i + \sum_k W_{i,k} [\mathbf{X}]_k$$

Re-index with $k = i + a$:

$$[\mathbf{H}]_i = u_i + \sum_a [\mathsf{V}]_{i,a} [\mathbf{X}]_{i+a}$$

**Step 2 — Impose translation invariance:** $[\mathsf{V}]_{i,a}$ must not depend on $i$, so $[\mathsf{V}]_{i,a} = [\mathbf{V}]_a$:

$$[\mathbf{H}]_i = u + \sum_a [\mathbf{V}]_a [\mathbf{X}]_{i+a}$$

**Step 3 — Impose locality:** zero out contributions beyond $|a| > \Delta$:

$$\boxed{[\mathbf{H}]_i = u + \sum_{a=-\Delta}^{\Delta} [\mathbf{V}]_a [\mathbf{X}]_{i+a}}$$

This is exactly a **1D convolution** — a filter $\mathbf{V}$ of size $2\Delta+1$ sliding along the time axis.

**Adding channels:** Real audio representations often have multiple channels (e.g. stereo, or multiple learned feature maps). With input channel $c$ and output channel $d$, the full multi-channel 1D convolution is:

$$[\mathsf{H}]_{i,d} = \sum_{a=-\Delta}^{\Delta} \sum_c [\mathsf{V}]_{a,c,d} \cdot [\mathsf{X}]_{i+a,c}$$

This is the direct 1D analogue of equation (7) from the notes — the sum is just missing the $b$ spatial dimension.

---

#### Part 3: Treating Audio with Vision Tools via the Spectrogram

The hint points to a very powerful idea. Raw audio is 1D, but a **spectrogram** converts it into a 2D time-frequency representation, making it amenable to standard 2D CNNs.

**How it works:** the Short-Time Fourier Transform (STFT) computes the frequency content over successive short windows:

$$S(i, f) = \left|\sum_{n} x(n) \cdot w(n - i)\, e^{-2\pi j f n}\right|^2$$

where $w$ is a windowing function, $i$ indexes time, and $f$ indexes frequency. The result is a 2D matrix of shape (time frames) × (frequency bins), and taking the log magnitude gives the log-spectrogram — visually similar to an image.

**Why 2D CNNs apply naturally:**

| Dimension | Image | Spectrogram |
|---|---|---|
| Axis 1 | Horizontal (x) | Time |
| Axis 2 | Vertical (y) | Frequency |
| Channels | RGB (3) | Typically 1, or multiple (e.g. mel channels) |

Both **locality** and **translation invariance** carry over cleanly:

- *Locality in time:* a phoneme or note spans a short time window — local in the time axis
- *Locality in frequency:* harmonics and formants cluster in nearby frequency bands
- *Translation invariance in time:* the same word spoken earlier or later should give the same representation
- *Translation invariance in frequency:* this is less perfect — it partially holds (for melody/pitch patterns) but breaks down for sounds whose identity depends on absolute frequency content.

**The practical upshot** is that architectures like VGG or ResNet, designed for images, can be directly applied to spectrograms with minimal modification. This is widely done in audio classification (e.g. environmental sound recognition, music genre classification, keyword spotting) — you simply feed the spectrogram in as a single-channel "image."

### Exercise 3

Why might translation invariance not be a good idea after all? Give an example.

#### The core tension

Translation invariance says: **"it doesn't matter where in the image a pattern appears — treat it the same way."** This is great for detecting *what* is in an image, but becomes a problem when *where* something is actually matters.

---

#### Example: Face Classification

Consider a face detector trained with translation invariance. The network learns to recognize individual parts — eyes, nose, mouth — regardless of where they appear.

Now consider these two images:

**Image A** — a normal face:
```
👁  👁
   👃
   👄
```

**Image B** — a scrambled face:
```
   👄
👁    👃
   👁
```

A strongly translation-invariant network sees **the same set of features** in both images — two eyes, a nose, a mouth — and may classify both as a valid face with equal confidence. But clearly Image B is wrong. The **spatial relationships** between parts carry critical information that pure translation invariance throws away.

---

#### More examples where location matters

| Task | Why location matters |
|---|---|
| Face recognition | Eyes should be *above* the mouth — relative positions distinguish a real face from a scrambled one |
| Medical imaging | A tumor in the lung vs. the liver is a completely different diagnosis |
| Satellite imagery | A river *next to* a city vs. *through* a city has different meaning |
| Digit recognition | The position of a stroke in "6" vs "9" is the entire difference between the two |

---

#### So what's the resolution?

Modern architectures handle this in a few ways:

- **Deeper layers** progressively lose strict translation invariance and capture more global spatial structure, since they aggregate information from larger receptive fields
- **Positional encodings** (common in Transformers) explicitly inject location information back into the representation
- **Capsule Networks** (Hinton et al.) were specifically proposed to address this — encoding not just *whether* a feature is present but *where* and in what *orientation*

---

### Exercise 4

Do you think that convolutional layers might also be applicable for text data? Which problems might you encounter with language?

#### Yes — CNNs do apply to text!

Text is a 1D sequence (like audio), so the 1D convolution we derived earlier applies directly. Each word (or character) is represented as a vector (via an embedding), giving a 2D input of shape (sequence length) × (embedding dimension). A 1D convolutional filter slides along the **sequence** axis, detecting local patterns.

This was actually quite popular before Transformers took over — models like TextCNN (Kim, 2014) used exactly this approach for sentence classification.

---

#### What do locality and translation invariance mean for text?

**Locality** maps naturally — meaningful linguistic patterns are often local:
- N-grams: "not good", "very happy", "New York"
- A filter of size $\Delta = 2$ captures trigrams, which are genuinely useful features

**Translation invariance** also maps reasonably well:
- The phrase "not good" carries negative sentiment whether it appears at the start or end of a sentence
- A CNN can detect this pattern wherever it occurs — just like detecting Waldo anywhere in an image

So far so good. But here's where language gets tricky.

---

#### Problems unique to language

**1. Word order and long-range dependencies**

Unlike pixels, words have **syntactic relationships** that span long distances. Consider:

> "The cat that the dog chased **was** scared."

The verb "was" agrees with "cat", not "dog" — but they're far apart. A local convolutional filter with small $\Delta$ completely misses this. Images rarely have analogous long-range dependencies at the low level where CNNs operate.

**2. Translation invariance breaks for meaning**

Word order is **semantically critical** in a way that spatial position in images is not:

> "Dog bites man" vs. "Man bites dog"

Same words, same local features — completely different meanings. A translation-invariant CNN treats these nearly identically. Contrast this with images: a dog in the top-left vs. bottom-right is still just a dog.

**3. Discrete vs. continuous inputs**

Pixels are continuous values — nearby pixel values are often similar, making local filters smooth and well-behaved. Words are **discrete symbols** — neighboring words in a sentence have no guaranteed semantic similarity. "The cat sat" and "The dog sat" differ by one word but could mean quite different things contextually.

**4. Variable-length and ambiguous context**

The meaning of a word often depends on context that can be arbitrarily far away:

> "I saw her duck." (Did she lower her head, or is it her pet?)

Resolving this ambiguity requires global context — the whole sentence or even surrounding sentences — which locality explicitly rules out.

---

#### How the field moved on

| Architecture | Handles long-range? | Translation invariant? | Used for text? |
|---|---|---|---|
| TextCNN | ✗ (local only) | ✓ | Yes, sentence classification |
| RNN/LSTM | ✓ (sequential) | ✗ | Yes, but slow |
| Transformer | ✓ (global attention) | ✗ (uses position encoding) | Dominant today |

---

### Exercise 5

What happens with convolutions when an object is at the boundary of an image?

#### The problem

Recall the convolutional layer:

$$[\mathbf{H}]_{i,j} = u + \sum_{a=-\Delta}^{\Delta} \sum_{b=-\Delta}^{\Delta} [\mathbf{V}]_{a,b} \cdot [\mathbf{X}]_{i+a, j+b}$$

When the filter is centered near the **edge** of the image, some of the required indices $i+a$ or $j+b$ **fall outside the image boundary** — those pixels don't exist!

---

#### A concrete example

Say we have a tiny 4×4 image and a filter of size $\Delta=1$ (so a 3×3 filter):

```
Image (4×4):
┌─────────────┐
│ 1  2  3  4  │
│ 5  6  7  8  │
│ 9  10 11 12 │
│ 13 14 15 16 │
└─────────────┘
```

When the filter is centered at pixel $(0, 0)$ (top-left corner), it tries to read:

```
[?,  ?,  ?]
[?,  1,  2]
[?,  5,  6]
```

The `?` positions are **outside the image** — what do we put there?

---

#### The three standard solutions

**1. Valid convolution (no padding)**

Simply don't compute outputs for positions where the filter goes out of bounds. Only compute where the filter fits **entirely inside** the image.

```
Input:  4×4
Filter: 3×3 (Δ=1)
Output: 2×2  ← shrinks!
```

- ✅ No made-up values
- ❌ Output is smaller than input — shrinks with every layer
- ❌ **Boundary pixels contribute to far fewer output values than center pixels** — information at edges is underrepresented

**2. Same/zero padding**

Pad the image boundary with **zeros** before applying the filter, so the output stays the same size as the input:

```
Padded image (6×6):
┌───────────────────┐
│ 0  0  0  0  0  0  │
│ 0  1  2  3  4  0  │
│ 0  5  6  7  8  0  │
│ 0  9  10 11 12 0  │
│ 0  13 14 15 16 0  │
│ 0  0  0  0  0  0  │
└───────────────────┘
```

- ✅ Output same size as input
- ✅ Simple and universally used
- ❌ Zeros are artificial — the network sees a "wall of nothing" at the border that doesn't exist in the real image

**3. Circular / reflective padding**

Instead of padding with zeros, **wrap around** (circular) or **mirror** the image:

Reflective padding mirrors the edge pixels:
```
┌───────────────────┐
│ 6  5  6  7  8  7  │
│ 2  1  2  3  4  3  │
│ 6  5  6  7  8  7  │  ← actual image rows
│ 10 9  10 11 12 11 │
│ 14 13 14 15 16 15 │
│ 10 9  10 11 12 11 │
└───────────────────┘
```

- ✅ More natural than zeros — no artificial boundary
- ✅ Common in image generation and style transfer tasks
- ❌ Still an approximation of what's "beyond" the edge

---

#### The deeper problem: boundary objects

Now back to the original question — what if an **object sits at the boundary**?

The fundamental issue is that the filter sees an **incomplete neighborhood** for boundary pixels. With zero padding, half the filter's view is artificial zeros rather than real image content. This means:

- The feature detector gets a **weaker, distorted signal** for objects near edges
- The network implicitly learns that "near the border" looks different from "in the center" for the same object
- This partially **breaks translation invariance** — a cat in the center and the same cat at the edge are processed differently, contradicting our original design principle!

---

#### Summary table

| Method | Output size | Boundary quality | Translation invariance |
|---|---|---|---|
| Valid (no padding) | Shrinks | Best (no artifacts) | Broken — edges ignored |
| Zero padding | Preserved | Artificial zeros | Approximately preserved |
| Reflective padding | Preserved | More natural | Better preserved |

---

### Exercise 6

Prove that the convolution is symmetric, i.e., $f * g=g * f$.

#### What we want to prove

Starting from the discrete 2D convolution definition:

$$(f * g)(i,j) = \sum_a \sum_b f(a,b) \cdot g(i-a, j-b)$$

We want to show this equals:

$$(g * f)(i,j) = \sum_a \sum_b g(a,b) \cdot f(i-a, j-b)$$

---

#### The proof

Start with the definition of $f * g$:

$$(f * g)(i,j) = \sum_a \sum_b f(a,b) \cdot g(i-a, j-b)$$

**Step 1 — Substitute variables.** Let $u = i - a$ and $v = j - b$, which means $a = i - u$ and $b = j - v$.

$$(f * g)(i,j) = \sum_u \sum_v f(i-u,\ j-v) \cdot g(u, v)$$

**Step 2 — Rearrange** (multiplication is commutative):

$$= \sum_u \sum_v g(u, v) \cdot f(i-u,\ j-v)$$

**Step 3 — Recognize the definition.** This is exactly the definition of $(g * f)(i,j)$:

$$= (g * f)(i,j)$$

Therefore:

$$\boxed{f * g = g * f}$$

$\blacksquare$

---

#### Intuition

Think of convolution as measuring the **overlap** between two functions as one slides over the other. Symmetry says: it doesn't matter which one you choose to slide — the overlap is the same either way. Whether you drag $g$ across $f$ or drag $f$ across $g$, you're measuring the same interaction.